# Inference Lab: Pricing Under Uncertainty


**Run this notebook in Google Colab.** The `demand` module is compiled for Linux and Python 3.13 only, so it will not import on a Mac or Windows machine.

The goal of this lab is to use what you've learned in this class to find the optimal price for something. You can't know for sure what the underlying curve of revenue vs price looks like. All you know is what you would know in the real world, information you gain by sampling-- you can market test any price using the demand function below. When you set the price, the market test runs for a certain amount of time, and then reports responses-- a list of 1s (bought) and 0s (didn't buy) for that price point. The total number in the sample, $n$, is therefore a random variable, given by `len(responses)`. The number of consumers who buy the product is `responses.sum().` Every time you run a market test, it takes time (literally, it will take a while to run, so don't do this lab last minute!).

In [9]:
!git clone -q https://github.com/smgroves/demand_lab.git

%cd /content/demand_lab

from demand import demand

responses = demand(price=14.0)

print(len(responses), responses.sum())

/content/demand_lab
145 117


## 1. Compare two prices

Run one market test with price = 30. Then test price = 25. Which price earns more revenue per potential customer?

1. For each price, estimate the proportion $\hat{p}$ of customers who buy, and the revenue per customer $\hat{r}$.
2. Estimate the standard error of each $\hat{r}$ by bootstrapping.
3. Compute a 95% confidence interval for each $\hat{r}$.
4. Compute a 95% confidence interval for the difference $\hat{r}_{25} - \hat{r}_{30}$.
   
5. Interpret the estimates, SEs, and intervals. Report which price earns more profit, and how confident you are.



In [10]:
import numpy as np
import pandas as pd

np.random.seed(5030)
R = 5_000
#Had some help from claude and gemini with some of the coding portions as well as reasonings for some of the outputs
def bootstrap_r(responses, price):
    n = len(responses)
    reps = []
    for t in range(R):
        idx = np.random.randint(0, n, n)
        reps.append(price * responses[idx].mean())
    return np.array(reps)

# one market test at each price
resp_30 = demand(price=30.0)
resp_25 = demand(price=25.0)

# p_hat and r_hat
p_30, p_25 = resp_30.mean(), resp_25.mean()
r_30, r_25 = 30.0 * p_30, 25.0 * p_25

# bootstrap standard errors
reps_30 = bootstrap_r(resp_30, 30.0)
reps_25 = bootstrap_r(resp_25, 25.0)
se_30 = np.std(reps_30, ddof=1)
se_25 = np.std(reps_25, ddof=1)

# 95% CIs: Normal approximation and percentile bootstrap
ci_30 = (r_30 - 1.96 * se_30, r_30 + 1.96 * se_30)
ci_25 = (r_25 - 1.96 * se_25, r_25 + 1.96 * se_25)
pct_30 = np.quantile(reps_30, [.025, .975])
pct_25 = np.quantile(reps_25, [.025, .975])

summary = pd.DataFrame({
    "n": [len(resp_30), len(resp_25)],
    "bought": [resp_30.sum(), resp_25.sum()],
    "p_hat": [p_30, p_25],
    "r_hat": [r_30, r_25],
    "SE": [se_30, se_25],
    "95% CI (Normal)": [f"({ci_30[0]:.2f}, {ci_30[1]:.2f})", f"({ci_25[0]:.2f}, {ci_25[1]:.2f})"],
    "95% CI (percentile)": [f"({pct_30[0]:.2f}, {pct_30[1]:.2f})", f"({pct_25[0]:.2f}, {pct_25[1]:.2f})"],
}, index=["$30", "$25"]).round(3)
print(summary)

# difference: the two samples are independent, so resample each and subtract
diff_hat = r_25 - r_30
reps_diff = reps_25 - reps_30
se_diff = np.std(reps_diff, ddof=1)
ci_diff = (diff_hat - 1.96 * se_diff, diff_hat + 1.96 * se_diff)
pct_diff = np.quantile(reps_diff, [.025, .975])

print(f"\nr_25 - r_30 = {diff_hat:.2f}, SE = {se_diff:.2f}")
print(f"95% CI (Normal):     [{ci_diff[0]:.2f}, {ci_diff[1]:.2f}]")
print(f"95% CI (percentile): [{pct_diff[0]:.2f}, {pct_diff[1]:.2f}]")

       n  bought  p_hat   r_hat     SE 95% CI (Normal) 95% CI (percentile)
$30  140      51  0.364  10.929  1.217   (8.54, 13.31)       (8.57, 13.29)
$25  169      68  0.402  10.059  0.952   (8.19, 11.93)       (8.28, 11.98)

r_25 - r_30 = -0.87, SE = 1.53
95% CI (Normal):     [-3.86, 2.12]
95% CI (percentile): [-3.82, 2.18]


**Estimates.** At price 30, 51 of the 140 customers bought, so p-hat = 0.364 and r-hat = 10.93 dollars per customer. At price 25, 68 of 169 had bought, so p-hat = 0.402 and r-hat = 10.06 dollars. The share of customers who buy is only a little lower at price 30, which is why 30 still looks slightly better: nearly as many people buy, but each one pays 5 more dollars.

**Standard errors and intervals.** Tthe bootstrap SEs are 1.22 at price 30 and 0.95 at price 25, which is about how much r-hat would move if I ran each market test again. The 95% CIs for revenue per customer are (8.54, 13.31) at price 30 and (8.19, 11.93) at price 25. There is quite a bit of some overlap, but overlap alone doesn't settle it, so I built an interval for the difference.

**Difference.** r-hat at price 25 minus r-hat at price 30 is -0.87, with a bootstrap SE of 1.53. The 95% CI for the difference is [-3.86, 2.12] (the percentile version is [-3.82, 2.18]). It contains 0, so I would fail to reject the null that the two prices earn the same.

**Conclusion.** Price 30 has the higherest estimate, by 0.87 dollars per customer, but with one test at each price I dont have enought evidence that it earns more. The data are consistent with price 30 being up to 3.86 dollars better, or price 25 being up to 2.12 dollars better, so I'm not confident either way. That doesn't mean the two prices earn the same, just that one test at each price isn't enough to tell. No cost is given so the profit here is just the revenue of the per customer.

## 2. The cost of sampling

7. Run a second market test at each price. Pool the two price = 30 samples (`np.concatenate`), and pool the two price = 25 samples.
8. Recompute the SEs, the intervals, and the interval for the difference. By roughly what factor did the SE shrink? Explain why, using the SE formula.
   
9.  **Was the extra sampling worth it?** Every market test takes time, and while you're testing you aren't selling at your best price, so each test costs the company revenue. Did the first round already settle the question (its difference interval excluded 0)? If so, the second round cost time and bought nothing but a narrower interval. If not, did the second round settle it?


In [11]:
# second market test at each price, pooled with the first
resp_30_2 = demand(price=30.0)
resp_25_2 = demand(price=25.0)

pool_30 = np.concatenate([resp_30, resp_30_2])
pool_25 = np.concatenate([resp_25, resp_25_2])

pool_r_30 = 30.0 * pool_30.mean()
pool_r_25 = 25.0 * pool_25.mean()

# bootstrap SEs and 95% CIs on the pooled samples
pool_reps_30 = bootstrap_r(pool_30, 30.0)
pool_reps_25 = bootstrap_r(pool_25, 25.0)
pool_se_30 = np.std(pool_reps_30, ddof=1)
pool_se_25 = np.std(pool_reps_25, ddof=1)

pool_ci_30 = (pool_r_30 - 1.96 * pool_se_30, pool_r_30 + 1.96 * pool_se_30)
pool_ci_25 = (pool_r_25 - 1.96 * pool_se_25, pool_r_25 + 1.96 * pool_se_25)
pool_pct_30 = np.quantile(pool_reps_30, [.025, .975])
pool_pct_25 = np.quantile(pool_reps_25, [.025, .975])

summary2 = pd.DataFrame({
    "n": [len(resp_30), len(pool_30), len(resp_25), len(pool_25)],
    "r_hat": [r_30, pool_r_30, r_25, pool_r_25],
    "SE": [se_30, pool_se_30, se_25, pool_se_25],
    "95% CI (Normal)": [f"({a:.2f}, {b:.2f})" for a, b in [ci_30, pool_ci_30, ci_25, pool_ci_25]],
    "95% CI (percentile)": [f"({a:.2f}, {b:.2f})" for a, b in [
        np.quantile(reps_30, [.025, .975]), pool_pct_30, np.quantile(reps_25, [.025, .975]), pool_pct_25]],
}, index=["$30 round 1", "$30 pooled", "$25 round 1", "$25 pooled"]).round(3)
print(summary2)

# difference r_25 - r_30 on the pooled samples
pool_diff_hat = pool_r_25 - pool_r_30
pool_reps_diff = pool_reps_25 - pool_reps_30
pool_se_diff = np.std(pool_reps_diff, ddof=1)
pool_ci_diff = (pool_diff_hat - 1.96 * pool_se_diff, pool_diff_hat + 1.96 * pool_se_diff)
pool_pct_diff = np.quantile(pool_reps_diff, [.025, .975])

print(f"\nround 1: r_25 - r_30 = {diff_hat:.2f}, SE = {se_diff:.2f}, 95% CI (Normal) [{ci_diff[0]:.2f}, {ci_diff[1]:.2f}]")
print(f"pooled:  r_25 - r_30 = {pool_diff_hat:.2f}, SE = {pool_se_diff:.2f}, 95% CI (Normal) [{pool_ci_diff[0]:.2f}, {pool_ci_diff[1]:.2f}], "
      f"percentile [{pool_pct_diff[0]:.2f}, {pool_pct_diff[1]:.2f}]")

# how much did the SE shrink?
n_mult = (len(pool_30) + len(pool_25)) / (len(resp_30) + len(resp_25))
print("\nSE ratio (pooled / round 1):")
print(f"  $30: {pool_se_30 / se_30:.2f}")
print(f"  $25: {pool_se_25 / se_25:.2f}")
print(f"  difference: {pool_se_diff / se_diff:.2f}")
print(f"  n grew by {n_mult:.1f}x, so 1/sqrt({n_mult:.1f}) = {1 / np.sqrt(n_mult):.2f}   (1/sqrt(2) = {1 / np.sqrt(2):.2f})")

               n   r_hat     SE 95% CI (Normal) 95% CI (percentile)
$30 round 1  140  10.929  1.217   (8.54, 13.31)       (8.57, 13.29)
$30 pooled   329  11.854  0.799  (10.29, 13.42)      (10.30, 13.50)
$25 round 1  169  10.059  0.952   (8.19, 11.93)       (8.28, 11.98)
$25 pooled   364  10.302  0.636   (9.06, 11.55)       (9.07, 11.54)

round 1: r_25 - r_30 = -0.87, SE = 1.53, 95% CI (Normal) [-3.86, 2.12]
pooled:  r_25 - r_30 = -1.55, SE = 1.02, 95% CI (Normal) [-3.55, 0.44], percentile [-3.56, 0.42]

SE ratio (pooled / round 1):
  $30: 0.66
  $25: 0.67
  difference: 0.67
  n grew by 2.2x, so 1/sqrt(2.2) = 0.67   (1/sqrt(2) = 0.71)


**Why the SE shrank.** The second test made my sample about 2.2 times bigger (n went from 140 to 329 at price 30, and from 169 to 364 at price 25), so the SE had gotten smaller. It dropped to 0.66 of what it was at price 30 and 0.67 at price 25, and the SE of the difference dropped to 0.67. That's about what the formula says: SE(r-hat) = P × sqrt(p-hat × (1 − p-hat) / n). Since n is under a square root, making it 2.2 times bigger only shrinks the SE to about 0.67 of what it was, not half. It's not exact because n is random and p-hat changed a bit with the new data. To cut the SE in half, I'd need four times as much data.

**Was the extra sampling worth it?** I don't really think it was. Neither round settled it. The first round's 95% CI for r-hat at price 25 minus r-hat at price 30 was [-3.86, 2.12], and the pooled CI is [-3.55, 0.44]. Both contain 0, so I fail to reject the null that the two prices earn the same. The second round did make the interval more narrowwer (width 5.98 to 3.99), and it moved the estimate a bit further toward price 30. The price 30 looked 0.87 dollars better after round 1 and 1.55 dollars better once it wasd pooled. Price 30 still has the slightly higher r-hat, but I can't rule out that the two prices earn the same. If the true gap is about what I measured, I'd need about 1.7 times as much data as I have now before the interval excluded 0, so one or two more tests at each price could settle it, but I think the extra tests would be better spent looking at other prices.

## 3. Finding the optimal price

Now find the price that maximizes revenue per customer. Search prices between about \$5 and \$100.

Every market test takes time, and time spent testing is revenue lost: while you test, you aren't selling at your best price. Your confidence intervals tell you whether another test is worth running. **Use as few market tests as you can (<25 would be good).** Your grade will be based on, in this order, whether you:
- accurately used the information you have by accurately interpreting confidence intervals
- found a good price, close to the optimal price
- and used fewer tests.

If this sounds fun to you, run with it! If you are still very confused about what to do, read the hints below.

<details><Summary>Hints</summary>
<b>Hint 1:</b> There are a few ways you could search:

- Brute force: test every price on a fine grid. This takes a market test at every price, including prices that are obviously bad. This would be a bad choice.
- Hill climbing: compare the CI for two prices, move toward the better one, and repeat. For example, start from the 25/30 comparison above; based on which price is better, choose another price \$10 away in that direction (15 if 25 was better; 40 if 30 was better). Shrink the step size as you get close to a good price. If you can't detect a difference between two prices, either shrink the step size or increase the sample size (i.e. run another market test at the same price).
- other strategies that I won't mention here.
  
<b>Hint 2:</b> You may want to try starting your tests with a few different prices across the range; there may be local maxima you will get stuck in if you just hill climb towards a single maximum.

<b>Hint 3:</b> You can make up your own strategy or try a mix of strategies. However, keep in mind that your grade will reflect how well you ran A/B testing and interpreting confidence intervals, more than how complex your strategy is. Similarly, a lucky guess with a brute force strategy would not be a good reproducible strategy.

<b>Hint 4:</b>
Turn your code from part 1 into a function, so you can easily input responses, run bootstrapping, calculate SE, and calculate confidence intervals. Then make another function where you give it two confidence intervals and make a decision based on them: sample more at the same test, or move in one direction with some step size.
   </details>

**Before you run any tests,** write down your plan: which prices you'll test first, how you'll decide where to go next, and what your intervals need to show before you stop. Then follow it, and note anywhere you changed course and why.

For any method, stop when your intervals show that another test is unlikely to change your answer by enough to matter.

**Deliverable:** A ~2-3 paragraph write up of the strategy you chose, your final recommended price, a 95% CI for $r$ at that price, the number of market tests you ran, and one or two sentences on how your intervals told you when to stop.

In [12]:
data = {30.0: [resp_30, resp_30_2], 25.0: [resp_25, resp_25_2]}
BUDGET = 24
LOW, HIGH = 5.0, 100.0
log = []

def run_test(price):
    data.setdefault(price, []).append(demand(price=price))

def tests_run():
    return sum(len(tests) for tests in data.values())

def r_hat(price):
    return price * np.concatenate(data[price]).mean()

def compare(a, b):
    reps_a = bootstrap_r(np.concatenate(data[a]), a)
    reps_b = bootstrap_r(np.concatenate(data[b]), b)
    d = r_hat(a) - r_hat(b)
    se = np.std(reps_a - reps_b, ddof=1)
    return d, d - 1.96 * se, d + 1.96 * se

def verdict(a, b):
    d, lo, hi = compare(a, b)
    log.append({"tests": tests_run(), "compare": f"${a:g} vs ${b:g}", "r_a - r_b": d, "CI low": lo, "CI high": hi})
    return 1 if lo > 0 else (-1 if hi < 0 else 0)

def table(prices):
    rows = []
    for P in prices:
        resp = np.concatenate(data[P])
        r = P * resp.mean()
        se = np.std(bootstrap_r(resp, P), ddof=1)
        rows.append({"price": P, "tests": len(data[P]), "n": len(resp), "r_hat": r, "SE": se,
                     "95% CI": f"({r - 1.96 * se:.2f}, {r + 1.96 * se:.2f})"})
    return pd.DataFrame(rows).set_index("price").round(2)

# coarse scan: one test at each price
for price in [10.0, 40.0, 55.0, 70.0, 85.0]:
    run_test(price)
scanned = sorted(data)
print(table(scanned))
print(f"market tests so far: {tests_run()}")

       tests    n  r_hat    SE          95% CI
price                                         
10.0       1  199   9.20  0.19    (8.81, 9.58)
25.0       2  364  10.30  0.65   (9.03, 11.57)
30.0       2  329  11.85  0.80  (10.29, 13.42)
40.0       1  188  14.68  1.40  (11.93, 17.43)
55.0       1  155  14.19  1.92  (10.43, 17.96)
70.0       1  186  17.69  2.19  (13.39, 21.99)
85.0       1  154  20.97  2.95  (15.19, 26.76)
market tests so far: 9


In [13]:
def climb(P, h, min_h=2.0):
    while h >= min_h and tests_run() < BUDGET:
        moved = False
        for cand in (P + h, P - h):
            if cand < LOW or cand > HIGH or tests_run() >= BUDGET:
                continue
            if cand not in data:
                run_test(cand)
            v = verdict(cand, P)
            if v == 0 and tests_run() + 2 <= BUDGET:
                run_test(cand)
                run_test(P)
                v = verdict(cand, P)
            if v > 0:
                P, moved = cand, True
                break
        if not moved:
            h = h / 2
    return P

# hill climb from the best scan price
P1 = climb(max(scanned, key=r_hat), 10.0)

# climb from any other scan price that is not clearly worse, then keep the better result
others = [P for P in scanned if abs(P - P1) > 4 and tests_run() < BUDGET and verdict(P, P1) >= 0]
if others:
    P2 = climb(max(others, key=r_hat), 10.0)
    v = verdict(P2, P1)
    if v > 0 or (v == 0 and r_hat(P2) > r_hat(P1)):
        P1 = P2

print(pd.DataFrame(log).round(2).to_string())
print(f"market tests: {tests_run()}")

    tests       compare  r_a - r_b  CI low  CI high
0      10    $95 vs $85      -9.93  -16.99    -2.87
1      11    $75 vs $85      -5.52  -12.92     1.88
2      13    $75 vs $85      -3.89   -9.02     1.25
3      14    $90 vs $85      -6.15  -12.60     0.30
4      16    $90 vs $85      -7.20  -11.80    -2.60
5      17    $80 vs $85      -0.13   -6.03     5.78
6      19    $80 vs $85      -1.19   -5.64     3.27
7      20  $87.5 vs $85      -1.45   -7.19     4.29
8      22  $87.5 vs $85       0.12   -4.43     4.67
9      23  $82.5 vs $85       4.35   -1.50    10.21
10     23    $10 vs $85     -10.15  -12.39    -7.92
11     23    $25 vs $85      -9.05  -11.58    -6.52
12     23    $30 vs $85      -7.50  -10.23    -4.77
13     23    $40 vs $85      -4.67   -8.20    -1.14
14     23    $55 vs $85      -5.16   -9.50    -0.82
15     23    $70 vs $85      -1.66   -6.65     3.33
16     23    $80 vs $70       1.60   -3.87     7.07
17     24    $60 vs $70      -3.32   -9.40     2.75
18     24   

In [14]:
final = max(data, key=r_hat)
resp_final = np.concatenate(data[final])
r_final = final * resp_final.mean()
se_final = np.std(bootstrap_r(resp_final, final), ddof=1)
ci_final = (r_final - 1.96 * se_final, r_final + 1.96 * se_final)

print(table(sorted(data)))
print(f"\nrecommended price: ${final:g}")
print(f"r_hat = ${r_final:.2f}, SE = ${se_final:.2f}, 95% CI [{ci_final[0]:.2f}, {ci_final[1]:.2f}]")
print(f"market tests run: {tests_run()}")

# is any tested price clearly better than the recommendation?
rows = []
for P in sorted(data):
    if P != final:
        d, lo, hi = compare(P, final)
        rows.append({"price": P, "r_P - r_final": d, "CI low": lo, "CI high": hi,
                     "clearly better?": "yes" if lo > 0 else "no"})
print("\nevery other tested price vs the recommendation")
print(pd.DataFrame(rows).set_index("price").round(2))

# draft write-up
below = max([P for P in data if P < final], default=None)
above = min([P for P in data if P > final], default=None)
nbrs = []
nbr_better = False
for P in (below, above):
    if P is not None:
        d, lo, hi = compare(P, final)
        label = "clearly worse" if hi < 0 else ("clearly better" if lo > 0 else "not clearly different")
        nbr_better = nbr_better or lo > 0
        nbrs.append(f"${P:g} ({label}; CI for r_{P:g} - r_{final:g} is [{lo:.2f}, {hi:.2f}])")

       tests    n  r_hat    SE          95% CI
price                                         
10.0       1  199   9.20  0.20    (8.81, 9.58)
25.0       2  364  10.30  0.65   (9.03, 11.57)
30.0       2  329  11.85  0.80  (10.29, 13.42)
40.0       1  188  14.68  1.41  (11.91, 17.45)
55.0       1  155  14.19  1.95  (10.38, 18.01)
60.0       1  142  14.37  2.13  (10.19, 18.54)
70.0       1  186  17.69  2.23  (13.31, 22.07)
75.0       2  386  17.10  1.60  (13.95, 20.24)
80.0       2  365  19.29  1.79  (15.78, 22.80)
82.5       1  181  23.70  2.78  (18.25, 29.16)
85.0       5  962  19.35  1.16  (17.08, 21.62)
87.5       2  328  19.47  2.01  (15.53, 23.42)
90.0       2  314  13.47  1.84   (9.86, 17.08)
95.0       1  215  11.05  2.07   (6.98, 15.11)

recommended price: $82.5
r_hat = $23.70, SE = $2.81, 95% CI [18.20, 29.20]
market tests run: 24

every other tested price vs the recommendation
       r_P - r_final  CI low  CI high clearly better?
price                                            

**Result.** My recommended price is 82.50 dollars, with r  hat = 23.70 dollars per customer, SE = 2.81 dollars and a 95% CI of [18.20, 29.20]. I ran 24 market tests in total, including the 4 from Parts 1 and 2. Because 82.5 was chosen as the highest r-hat among 14 tested prices, this interval is probably optimistic. It comes from one test (n = 181), while the prices on both sides, 80 and 85, came out around 19.3 with more data behind them (n = 365 and n = 962), so the true revenue at 82.5 is probably lower than 23.70.

**When I stopped.** I stopped after 24 tests, when I reached my budget of 24. The nearest tested prices on each side of 82.5 were 80 and 85. The 95% CI for the difference in revenue per customer between 80 and 82.5 was [-11.01, 2.18], and between 85 and 82.5 was [-10.18, 1.48]. Both of the intervals had 0, so neither neighbor is shown to actually surely be better, and another test is unlikely to change my answer by enough to actually matter. The intervals are still wide, though: the SE of r-hat at 82.5 is 2.81 dollars (1.79 at 80 and 1.16 at 85), which is too large to separate prices that are within a few dollars of each other.

**When I stopped.** I stopped after {number of tests} tests, when I reached my budget of 24. The nearest tested prices on each side of \${final price} were \${price below} and \${price above}. The 95% CI for the difference in revenue per customer between \${price below} and \${final price} was [{low}, {high}], and between \${price above} and \${final price} was [{low}, {high}]. Both intervals contain 0, so neither neighbor is shown to be better, and another test is unlikely to change my answer by enough to matter. The intervals are still wide, though: with one or two tests per price, the SE of r aht is around \$1 to \$2, which is too large to separate prices that are within a few dollars of each other.